In [1]:
from utils.config import load_config
from analysis.target_analysis import analyze_numeric_feature
from analysis.numeric.numeric_summary import build_numeric_summary
from feature_utils.feature_engineering import replace_with_has_large
from feature_utils.data_cleaning import fill_missing_values
import pandas as pd
%matplotlib inline

In [2]:
config = load_config()
train_df = pd.read_csv(config["paths"]["train"])
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
summary = build_numeric_summary(train_df)
summary

,feature,mean,median,std,min,max,missing,missing_%,has_outliers,skew
2,LotFrontage,70.049958,69.0,24.284752,21.0,313.0,True,0.177397,True,2.163569
25,GarageYrBlt,1978.506164,1980.0,24.689725,1900.0,2010.0,True,0.055479,False,-0.649415
8,MasVnrArea,103.685262,0.0,181.066207,0.0,1600.0,True,0.005479,True,2.669084
3,LotArea,10516.828082,9478.5,9981.264932,1300.0,215245.0,False,0.000000,True,12.207688
1,MSSubClass,56.897260,50.0,42.300571,20.0,190.0,False,0.000000,True,1.407657
0,Id,730.500000,730.5,421.610009,1.0,1460.0,False,0.000000,False,0.000000
5,OverallCond,5.575342,5.0,1.112799,1.0,9.0,False,0.000000,True,0.693067
4,OverallQual,6.099315,6.0,1.382997,1.0,10.0,False,0.000000,True,0.216944
7,YearRemodAdd,1984.865753,1994.0,20.645407,1950.0,2010.0,False,0.000000,False,-0.503562
6,YearBuilt,1971.267808,1973.0,30.202904,1872.0,2010.0,False,0.000000,True,-0.613461


## Feature Engineering: sparse numerical features

В процессе анализа числовых признаков было обнаружено, что некоторые из них имеют сильный перекос распределения (high skew) и медиану, равную 0.

Это означает:
- большинство значений в этих признаках равно 0
- небольшое количество объектов имеет ненулевые (иногда большие) значения

К таким признакам относятся:
- `MiscVal`
- `PoolArea`
- `3SsnPorch`
- `LowQualFinSF`

### Интерпретация

Для данных признаков более важен **факт наличия**, чем конкретное значение.

Например:
- наличие бассейна (`PoolArea > 0`) важнее его площади
- наличие дополнительных построек (`MiscVal > 0`) важнее их стоимости

### Принятое решение

Преобразовать данные признаки в бинарный формат:
- `1` — признак присутствует
- `0` — отсутствует

### Причины такого подхода

- уменьшение влияния выбросов
- упрощение модели
- повышение устойчивости к шуму
- улучшение интерпретируемости

### Результат

Каждый признак заменяется на новый бинарный:
- `Has_MiscVal`
- `Has_PoolArea`
- `Has_3SsnPorch`
- `Has_LowQualFinSF`

Оригинальные признаки удаляются из датасета.

In [4]:
features_sparse = [
    'MiscVal',
    'PoolArea',
    '3SsnPorch',
    'LowQualFinSF'
]

train_df = replace_with_has_large(train_df, features_sparse)

In [5]:
summary = build_numeric_summary(train_df)
summary

,feature,mean,median,std,min,max,missing,missing_%,has_outliers,skew
2,LotFrontage,70.049958,69.0,24.284752,21.0,313.0,True,0.177397,True,2.163569
24,GarageYrBlt,1978.506164,1980.0,24.689725,1900.0,2010.0,True,0.055479,False,-0.649415
8,MasVnrArea,103.685262,0.0,181.066207,0.0,1600.0,True,0.005479,True,2.669084
3,LotArea,10516.828082,9478.5,9981.264932,1300.0,215245.0,False,0.000000,True,12.207688
1,MSSubClass,56.897260,50.0,42.300571,20.0,190.0,False,0.000000,True,1.407657
0,Id,730.500000,730.5,421.610009,1.0,1460.0,False,0.000000,False,0.000000
5,OverallCond,5.575342,5.0,1.112799,1.0,9.0,False,0.000000,True,0.693067
4,OverallQual,6.099315,6.0,1.382997,1.0,10.0,False,0.000000,True,0.216944
7,YearRemodAdd,1984.865753,1994.0,20.645407,1950.0,2010.0,False,0.000000,False,-0.503562
6,YearBuilt,1971.267808,1973.0,30.202904,1872.0,2010.0,False,0.000000,True,-0.613461


# Обработка пропусков (Missing Values)

## 📊 Анализ данных

В ходе EDA были найдены признаки с пропусками:

| Feature        | Missing % | Skew   | Outliers |
|---------------|----------|--------|----------|
| LotFrontage   | 17.7%    | 2.16   | ✅       |
| GarageYrBlt   | 5.5%     | -0.64  | ❌       |
| MasVnrArea    | 0.5%     | 2.66   | ✅       |

---

## 🧠 Подход к заполнению

При выборе стратегии использовались следующие правила:

1. Если распределение **смещено (skew > 1)** → используем **median**
2. Если распределение близко к нормальному → можно использовать **mean**
3. Если есть **выбросы** → median предпочтительнее
4. Если признак имеет **смысловую интерпретацию (например, 0 = отсутствие)** → важно учитывать это

In [7]:
df = fill_missing_values(
    train_df,
    columns=["LotFrontage", "GarageYrBlt", "MasVnrArea"],
    strategy="median",
)

In [8]:
summary = build_numeric_summary(train_df)
summary

,feature,mean,median,std,min,max,missing,missing_%,has_outliers,skew
0,Id,730.500000,730.5,421.610009,1.0,1460.0,False,0.0,False,0.000000
1,MSSubClass,56.897260,50.0,42.300571,20.0,190.0,False,0.0,True,1.407657
2,LotFrontage,69.863699,69.0,22.027677,21.0,313.0,False,0.0,True,2.409147
3,LotArea,10516.828082,9478.5,9981.264932,1300.0,215245.0,False,0.0,True,12.207688
4,OverallQual,6.099315,6.0,1.382997,1.0,10.0,False,0.0,True,0.216944
5,OverallCond,5.575342,5.0,1.112799,1.0,9.0,False,0.0,True,0.693067
6,YearBuilt,1971.267808,1973.0,30.202904,1872.0,2010.0,False,0.0,True,-0.613461
7,YearRemodAdd,1984.865753,1994.0,20.645407,1950.0,2010.0,False,0.0,False,-0.503562
8,MasVnrArea,103.117123,0.0,180.731373,0.0,1600.0,False,0.0,True,2.677616
9,BsmtFinSF1,443.639726,383.5,456.098091,0.0,5644.0,False,0.0,True,1.685503
